In [ ]:
!pip install scikit-optimize

## Data Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# load datasets
df_dss = pd.read_excel('dss.xlsx', header=0)
df_sss = pd.read_excel('sss.xlsx', header=0)
df_sts = pd.read_excel('sts.xlsx', header=0)
df_all = pd.read_excel('all-data.xlsx', header=0)

In [ ]:
df_dss

,Cement (kg/m3),FAF (Kg/m3),UFAF (Kg/m3),FAC (Kg/m3),GBFS (Kg/m3),Interval time (min),Age (days),DSS (MPa)
0,500,0,0,0,0,0,7,13.8
1,500,0,0,0,0,40,7,13.3
2,500,0,0,0,0,80,7,9.3
3,500,0,0,0,0,120,7,7.9
4,500,0,0,0,0,160,7,5.4
...,...,...,...,...,...,...,...,...
250,200,0,0,0,300,0,90,14.3
251,200,0,0,0,300,40,90,13.7
252,200,0,0,0,300,80,90,12.1
253,200,0,0,0,300,120,90,7.9


In [ ]:
# mix 3 dataset labels
df_main = df_dss.copy(deep=True)
df_main['SSS (MPa)'] = df_sss['SSS (MPa)']
df_main['STS (MPa)'] = df_sts['STS (MPa)']
df_main

,Cement (kg/m3),FAF (Kg/m3),UFAF (Kg/m3),FAC (Kg/m3),GBFS (Kg/m3),Interval time (min),Age (days),DSS (MPa),SSS (MPa),STS (MPa)
0,500,0,0,0,0,0,7,13.8,28.00,4.1088
1,500,0,0,0,0,40,7,13.3,26.10,3.7236
2,500,0,0,0,0,80,7,9.3,23.90,3.4026
3,500,0,0,0,0,120,7,7.9,21.60,3.2100
4,500,0,0,0,0,160,7,5.4,20.50,2.7606
...,...,...,...,...,...,...,...,...,...,...
250,200,0,0,0,300,0,90,14.3,26.95,3.2000
251,200,0,0,0,300,40,90,13.7,26.10,3.0000
252,200,0,0,0,300,80,90,12.1,25.60,1.9902
253,200,0,0,0,300,120,90,7.9,25.00,1.8618


In [ ]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255 entries, 0 to 254
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Cement (kg/m3)       255 non-null    int64  
 1   FAF (Kg/m3)          255 non-null    int64  
 2   UFAF (Kg/m3)         255 non-null    int64  
 3   FAC (Kg/m3)          255 non-null    int64  
 4   GBFS (Kg/m3)         255 non-null    int64  
 5   Interval time (min)  255 non-null    int64  
 6   Age (days)           255 non-null    int64  
 7   DSS (MPa)            255 non-null    float64
 8   SSS (MPa)            255 non-null    float64
 9   STS (MPa)            255 non-null    float64
dtypes: float64(3), int64(7)
memory usage: 20.1 KB


In [ ]:
# dataset statistics
df_main.describe()

,Cement (kg/m3),FAF (Kg/m3),UFAF (Kg/m3),FAC (Kg/m3),GBFS (Kg/m3),Interval time (min),Age (days),DSS (MPa),SSS (MPa),STS (MPa)
count,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000
mean,323.529412,44.117647,44.117647,44.117647,44.117647,80.000000,41.666667,12.046275,24.615490,3.453502
std,92.723947,89.507103,89.507103,89.507103,89.507103,56.679789,35.305008,4.658915,6.633486,1.536138
min,200.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.000000,2.000000,9.500000,0.834600
25%,275.000000,0.000000,0.000000,0.000000,0.000000,40.000000,7.000000,8.300000,20.150000,2.439600
50%,350.000000,0.000000,0.000000,0.000000,0.000000,80.000000,28.000000,12.200000,25.100000,3.081600
75%,425.000000,0.000000,0.000000,0.000000,0.000000,120.000000,90.000000,15.150000,29.275000,4.104400
max,500.000000,300.000000,300.000000,300.000000,300.000000,160.000000,90.000000,25.300000,42.800000,8.500000


In [ ]:
# check for NaN values
print(df_main.isnull().values.any())

False


In [ ]:
X = df_main.loc[:, :"Age (days)"]
Y = df_main.loc[:, "DSS (MPa)":"STS (MPa)"]


fig, axes = plt.subplots(nrows=3, ncols=7, figsize=(21, 9), dpi=1200)
plt.subplots_adjust(hspace=0.4, wspace=0.4)

for i, output_col in enumerate(Y.columns):
    for j, feature_col in enumerate(X.columns):
        sns.kdeplot(
            x=X[feature_col],
            y=Y[output_col],
            fill=True,
            ax=axes[i, j]
        )
        axes[i, j].set_xlabel(feature_col, fontweight='bold')
        axes[i, j].set_ylabel(output_col, fontweight='bold')

plt.show()

In [ ]:
# # Histogram of all data columns
# n_col = 5
# n_row = 2
# fig, axes = plt.subplots(n_row, n_col, figsize=(16, 6))
# axes = axes.flatten()

# for i, col in enumerate(df_main.columns):
#     sns.histplot(data=df_main, x=col, ax=axes[i], kde=True, color='orange')
#     axes[i].set_title(f'Distribution of {col}')

# plt.tight_layout()
# plt.show()

In [ ]:
# Correlation Matrix (Heatmap) of dataset
plt.figure(figsize=(10, 8), dpi=1200)
cor_matrix = df_main.corr()
sns.heatmap(cor_matrix,
            annot=True,
            cmap='Spectral',
            fmt=".2f",
            center=0,
            square=True)
plt.title('Correlation Matrix Heatmap')
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.tree import plot_tree

X_train, X_test, y_train, y_test = train_test_split(df_main.loc[:, :"Age (days)"], df_main.loc[:, "DSS (MPa)":"STS (MPa)"], test_size=0.3, random_state=42)

reg = DecisionTreeRegressor(max_depth=2, random_state=42)
reg.fit(X_train, y_train)

plt.figure(figsize=(30, 8), dpi=1200)
feature_names = X_train.columns.to_list()
plot_tree(reg,
          feature_names=feature_names,
          filled=True,
          rounded=True,
          fontsize=14)
plt.title("Tree Plot")
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.tree import plot_tree

X_train, X_test, y_train, y_test = train_test_split(df_main.loc[:, :"Age (days)"], df_main.loc[:, "DSS (MPa)":"STS (MPa)"], test_size=0.3, random_state=42)

reg = DecisionTreeRegressor(max_depth=2, random_state=42)
reg.fit(X_train, y_train)

importances = reg.feature_importances_
feature_names = X_train.columns
feature_importance_df = pd.DataFrame({
  'Feature': feature_names,
  'Importance': importances
})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
plt.figure(figsize=(10, 6), dpi=1200)
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='orange')
plt.title('Feature Importance')
plt.show()

In [ ]:
# # check feature importance with Decision Tree for 3 different outputs
# from sklearn.tree import DecisionTreeRegressor
# from sklearn.model_selection import train_test_split
# from sklearn.tree import plot_tree

# X_train_dss, X_test_dss, y_train_dss, y_test_dss = train_test_split(df_dss.loc[:, :"Age (days)"], df_dss['DSS (MPa)'], test_size=0.3, random_state=42)
# X_train_sss, X_test_sss, y_train_sss, y_test_sss = train_test_split(df_sss.loc[:, :"Age (days)"], df_sss['SSS (MPa)'], test_size=0.3, random_state=42)
# X_train_sts, X_test_sts, y_train_sts, y_test_sts = train_test_split(df_sts.loc[:, :"Age (days)"], df_sts['STS (MPa)'], test_size=0.3, random_state=42)

# regressor_dss = DecisionTreeRegressor(max_depth=4, random_state=42)
# regressor_sss = DecisionTreeRegressor(max_depth=4, random_state=42)
# regressor_sts = DecisionTreeRegressor(max_depth=4, random_state=42)

# regressor_dss.fit(X_train_dss, y_train_dss)
# regressor_sss.fit(X_train_sss, y_train_sss)
# regressor_sts.fit(X_train_sts, y_train_sts)

DecisionTreeRegressor(max_depth=4, random_state=42)

In [ ]:
# importances = regressor_dss.feature_importances_
# feature_names = X_train_dss.columns
# feature_importance_df = pd.DataFrame({
#   'Feature': feature_names,
#   'Importance': importances
# })
# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
# plt.figure(figsize=(10, 6), dpi=1200)
# plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='orange')
# plt.title('DSS Feature Importance')
# plt.show()

In [ ]:
# importances = regressor_sss.feature_importances_
# feature_names = X_train_sss.columns
# feature_importance_df = pd.DataFrame({
#   'Feature': feature_names,
#   'Importance': importances
# })
# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
# plt.figure(figsize=(10, 6), dpi=1200)
# plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='orange')
# plt.title('SSS Feature Importance')
# plt.show()

In [ ]:
# importances = regressor_sts.feature_importances_
# feature_names = X_train_sts.columns
# feature_importance_df = pd.DataFrame({
#   'Feature': feature_names,
#   'Importance': importances
# })
# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
# plt.figure(figsize=(10, 6), dpi=1200)
# plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='orange')
# plt.title('STS Feature Importance')
# plt.show()

In [ ]:
# plt.figure(figsize=(30, 8), dpi=1200)
# feature_names = X_train_dss.columns.to_list()
# plot_tree(regressor_dss,
#           feature_names=feature_names,
#           filled=True,
#           rounded=True,
#           fontsize=8)
# plt.title("Tree Plot of DSS")
# plt.show()

In [ ]:
# plt.figure(figsize=(30, 8), dpi=1200)
# feature_names = X_train_sss.columns.to_list()
# plot_tree(regressor_sss,
#           feature_names=feature_names,
#           filled=True,
#           rounded=True,
#           fontsize=8)
# plt.title("Tree Plot of SSS")
# plt.show()

In [ ]:
# plt.figure(figsize=(30, 8), dpi=1200)
# feature_names = X_train_sts.columns.to_list()
# plot_tree(regressor_sts,
#           feature_names=feature_names,
#           filled=True,
#           rounded=True,
#           fontsize=8)
# plt.title("Tree Plot of STS")
# plt.show()

In [ ]:
# # convert Non-numerical values to numerical ones
# from sklearn.preprocessing import LabelEncoder

# le = LabelEncoder()
# df_all['Test type'] = le.fit_transform(df_all['Test type'])
# df_all['Test type'].value_counts()
# X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(df_all.loc[:, :"Test type"], df_all['Strength (Mpa)'], test_size=0.3, random_state=42)

# regressor_all = DecisionTreeRegressor(max_depth=4, random_state=42)
# regressor_all.fit(X_train_all, y_train_all)

# plt.figure(figsize=(30, 8))
# feature_names = X_train_all.columns.to_list()
# plot_tree(regressor_dss,
#           feature_names=feature_names,
#           filled=True,
#           rounded=True,
#           fontsize=8)
# plt.title("Tree Plot of Mix Data")
# plt.show()

## Base 3 Output Models

In [ ]:
# Model Training For 3 different Output
from sklearn.linear_model import ElasticNet,Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.neural_network import MLPRegressor
import xgboost
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# train and test split
output_col = df_main.loc[:, "DSS (MPa)":"STS (MPa)"].columns.to_list()
X_train, X_test, y_train, y_test = train_test_split(df_main.loc[:, :"Age (days)"], df_main.loc[:, "DSS (MPa)":"STS (MPa)"], test_size=0.3, random_state=42)

# normalize data
scaler = StandardScaler()
scaler.set_output(transform='pandas')
scaler.fit(X_train.loc[:, :"Age (days)"])
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# def evaluate(model_name, y_pred):
#   print("*" * 10, model_name, "*" * 10)
#   for i in range(len(output_col)):
#     print("======", output_col[i], "======")
#     print("R2:", r2_score(y_test[output_col[i]], y_pred[:, i]))
#     print("MAE:", mean_absolute_error(y_test[output_col[i]], y_pred[:, i]))
#     print("MSE:", mean_squared_error(y_test[output_col[i]], y_pred[:, i]))
#     print("RMSE:", np.sqrt(mean_squared_error(y_test[output_col[i]], y_pred[:, i])))

models = {
    "ElasticNet": MultiOutputRegressor(ElasticNet()),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "Ridge": MultiOutputRegressor(Ridge()),
    "SVR": MultiOutputRegressor(SVR(kernel='rbf')),
    "GPR": MultiOutputRegressor(GaussianProcessRegressor()),
    "MLP": MultiOutputRegressor(MLPRegressor(max_iter=2000, random_state=4)),
    "RandomForest": RandomForestRegressor(random_state=42),
    "XGBoost": MultiOutputRegressor(xgboost.XGBRegressor(random_state=42)),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Gradient Boosting Reg": MultiOutputRegressor(GradientBoostingRegressor(random_state=42))
}

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

# create df for each output
def evaluate_df(model_name, y_pred):
    data = []
    for i, output in enumerate(output_col):
        r2 = r2_score(y_test[output], y_pred[:, i])
        mae = mean_absolute_error(y_test[output], y_pred[:, i])
        mse = mean_squared_error(y_test[output], y_pred[:, i])
        rmse = np.sqrt(mse)
        data.append([model_name, output, r2, mae, mse, rmse])
    df = pd.DataFrame(data, columns=["Model", "Output", "R2", "MAE", "MSE", "RMSE"])
    return df

# overall metrics
def compute_overall_metrics(model_name, y_pred):
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    return pd.DataFrame({
        "Model": [model_name],
        "R2": [r2],
        "MAE": [mae],
        "MSE": [mse],
        "RMSE": [rmse]
    })

# train models
metrics_list = []
metrics_list_overall = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=ConvergenceWarning)
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        df_metrics = evaluate_df(name, y_pred)

        metrics_list.append(df_metrics)

        metrics_list_overall.append(compute_overall_metrics(name, y_pred))

# Combine all results for each model
metrics_all = pd.concat(metrics_list, axis=0).reset_index(drop=True)

# Combine all models overall and sort based on R2
metrics_overall = pd.concat(metrics_list_overall, axis=0).reset_index(drop=True)
metrics_sorted = metrics_overall.sort_values(by="R2", ascending=False)

In [ ]:
print(metrics_all)

                    Model     Output        R2       MAE        MSE      RMSE
0              ElasticNet  DSS (MPa)  0.659741  2.122175   6.724069  2.593081
1              ElasticNet  SSS (MPa)  0.651107  3.011440  14.216766  3.770513
2              ElasticNet  STS (MPa)  0.138354  1.061437   2.018209  1.420637
3                     KNN  DSS (MPa)  0.820669  1.578182   3.543875  1.882518
4                     KNN  SSS (MPa)  0.694753  2.774416  12.438279  3.526794
5                     KNN  STS (MPa)  0.822913  0.459651   0.414786  0.644039
6                   Ridge  DSS (MPa)  0.796858  1.568000   4.014425  2.003603
7                   Ridge  SSS (MPa)  0.786149  2.358323   8.714025  2.951953
8                   Ridge  STS (MPa)  0.678658  0.653500   0.752671  0.867566
9                     SVR  DSS (MPa)  0.853890  1.355183   2.887364  1.699224
10                    SVR  SSS (MPa)  0.682332  2.488731  12.944386  3.597831
11                    SVR  STS (MPa)  0.901377  0.348617   0.231

In [ ]:
print("====== Three output =======")
print(metrics_sorted)

====== Three output =======
                   Model        R2       MAE       MSE      RMSE
9  Gradient Boosting Reg  0.961701  0.610707  0.783590  0.885206
7                XGBoost  0.940950  0.714608  1.209066  1.099575
6           RandomForest  0.926814  0.860906  1.458461  1.207668
4                    GPR  0.904964  1.014866  2.290336  1.513386
5                    MLP  0.874071  1.231470  3.652619  1.911183
8          Decision Tree  0.872347  1.061324  2.456466  1.567312
3                    SVR  0.812533  1.397510  5.354251  2.313925
1                    KNN  0.779445  1.604083  5.465647  2.337872
2                  Ridge  0.753888  1.526608  4.493707  2.119837
0             ElasticNet  0.483067  2.065018  7.653015  2.766408


## Mix Data

In [ ]:
df_all.head(1)

,Cement (kg/m3),FAF (Kg/m3),UFAF (Kg/m3),FAC (Kg/m3),GBFS (Kg/m3),Interval time (min),Age (days),Test type,Strength (Mpa)
0,500,0,0,0,0,0,7,1,28.0


In [ ]:
# Model Training for Mix Data
# train and test split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_all['Test type'] = le.fit_transform(df_all['Test type'])
df_all['Test type'].value_counts()

X_train, X_test, y_train, y_test = train_test_split(df_all.loc[:, :"Test type"], df_all['Strength (Mpa)'], test_size=0.3, random_state=42)

# normalize data
scaler = StandardScaler()
scaler.set_output(transform='pandas')
scaler.fit(df_all.loc[:, :"Test type"])
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

models = {
    "ElasticNet": ElasticNet(random_state=42),
    "Ridge": Ridge(),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "SVR": SVR(kernel='rbf'),
    "GPR": GaussianProcessRegressor(),
    "MLP": MLPRegressor(max_iter=3000, random_state=4),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "Gradient Boosting Reg": GradientBoostingRegressor(random_state=42),
    "XGBoost": xgboost.XGBRegressor(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    results.append({
        "Model": name,
        "R2": r2,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse
    })

metrics_df_all = pd.DataFrame(results).sort_values(by="R2", ascending=False).reset_index(drop=True)
print("====== Mix Data =======")
print(metrics_df_all)

====== Mix Data =======
                   Model        R2       MAE        MSE      RMSE
0                XGBoost  0.989959  0.666092   0.953428  0.976437
1                    MLP  0.988179  0.799519   1.122379  1.059424
2           RandomForest  0.985183  0.793066   1.406921  1.186137
3  Gradient Boosting Reg  0.984378  0.869157   1.483274  1.217898
4                    GPR  0.982309  0.930418   1.679747  1.296050
5           DecisionTree  0.979198  0.983222   1.975109  1.405386
6                    KNN  0.780105  3.262357  20.879076  4.569363
7                    SVR  0.425777  4.701700  54.522595  7.383942
8                  Ridge  0.250306  7.577406  71.183511  8.437032
9             ElasticNet  0.192089  7.663331  76.711228  8.758495


In [ ]:
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import warnings
warnings.filterwarnings("ignore", message="The objective has been evaluated at point")

In [ ]:
# Bayesian OPT on mix model with ~750 data samples

result_mix = []
n_iter = 100

elasticnet = ElasticNet(random_state=42)
elasticnet_p = {'l1_ratio': Real(0.0, 1.0), 'alpha': Real(0.01, 1.0, prior='log-uniform')}
opt = BayesSearchCV(elasticnet, search_spaces = elasticnet_p, cv = 5, n_iter = n_iter, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
opt.fit(X_train, y_train)
y_pred = opt.predict(X_test)
result_mix.append({
    "Model": "Elasticnet",
    "Best Params": opt.best_params_,
    "Test_MSE": mean_squared_error(y_test, y_pred)
})
print("OPT Elasticnet Done...")

knn = KNeighborsRegressor()
knn_p = {'n_neighbors': Integer(3, 20), 'weights': Categorical(['uniform', 'distance'])}
opt = BayesSearchCV(knn, search_spaces = knn_p, cv = 5, n_iter = n_iter, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
opt.fit(X_train, y_train)
y_pred = opt.predict(X_test)
result_mix.append({
    "Model": "KNN",
    "Best Params": opt.best_params_,
    "Test_MSE": mean_squared_error(y_test, y_pred)
})
print("OPT KNN Done...")

svr = SVR()
svr_p = {'kernel': Categorical(['linear', 'poly', 'rbf']), 'C': Real(0.01, 1.0, prior='log-uniform')}
opt = BayesSearchCV(svr, search_spaces = svr_p, cv = 5, n_iter = n_iter, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
opt.fit(X_train, y_train)
y_pred = opt.predict(X_test)
result_mix.append({
    "Model": "SVR",
    "Best Params": opt.best_params_,
    "Test_MSE": mean_squared_error(y_test, y_pred)
})
print("OPT SVR Done...")

dt = DecisionTreeRegressor(random_state=42)
dt_p = {'criterion': Categorical(["squared_error", "absolute_error"])}
opt = BayesSearchCV(dt, search_spaces = dt_p, cv = 5, n_iter = n_iter, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
opt.fit(X_train, y_train)
y_pred = opt.predict(X_test)
result_mix.append({
    "Model": "Decision Tree",
    "Best Params": opt.best_params_,
    "Test_MSE": mean_squared_error(y_test, y_pred)
})
print("OPT DT Done...")

rf = RandomForestRegressor(random_state=42)
rf_p = {'n_estimators': Integer(100, 500), 'criterion': Categorical(["squared_error", "absolute_error"])}
opt = BayesSearchCV(rf, search_spaces = rf_p, cv = 5, n_iter = n_iter, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
opt.fit(X_train, y_train)
y_pred = opt.predict(X_test)
result_mix.append({
    "Model": "Random Forest",
    "Best Params": opt.best_params_,
    "Test_MSE": mean_squared_error(y_test, y_pred)
})
print("OPT RF Done...")

gbr = GradientBoostingRegressor()
gbr_p = {'n_estimators': Integer(100, 500), 'learning_rate': Real(0.01, 0.3, prior='log-uniform')}
opt = BayesSearchCV(gbr, search_spaces = gbr_p, cv = 5, n_iter = n_iter, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
opt.fit(X_train, y_train)
y_pred = opt.predict(X_test)
result_mix.append({
    "Model": "Gradient Boosting Reg",
    "Best Params": opt.best_params_,
    "Test_MSE": mean_squared_error(y_test, y_pred)
})
print("OPT GBR Done...")

xgb = xgboost.XGBRegressor()
xgb_p = {'learning_rate': Real(0.01, 3.0, prior='log-uniform')}
opt = BayesSearchCV(xgb, search_spaces = xgb_p, cv = 5, n_iter = n_iter, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
opt.fit(X_train, y_train)
y_pred = opt.predict(X_test)
result_mix.append({
    "Model": "Xgboost",
    "Best Params": opt.best_params_,
    "Test_MSE": mean_squared_error(y_test, y_pred)
})
print("OPT Xgboost Done...")

for i in [(100,), (128,64),(128,64,32), (100, 50)]:
  mlp = MLPRegressor(hidden_layer_sizes=i, random_state=4)
  mlp_p = {
      'activation': Categorical(['tanh', 'relu']),
      'learning_rate_init': Real(0.0001, 0.1, prior='log-uniform')
      }
  opt = BayesSearchCV(mlp, search_spaces = mlp_p, cv = 5, n_iter = n_iter, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
  opt.fit(X_train, y_train)
  y_pred = opt.predict(X_test)
  result_mix.append({
      "Model": f"MLP {i}",
      "Best Params": opt.best_params_,
      "Test_MSE": mean_squared_error(y_test, y_pred)
  })
  print(f"OPT MLP {i} Done...")

OPT Elasticnet Done...
OPT KNN Done...
OPT SVR Done...
OPT DT Done...
OPT RF Done...
OPT GBR Done...
OPT Xgboost Done...
OPT MLP (100,) Done...
OPT MLP (128, 64) Done...
OPT MLP (128, 64, 32) Done...
OPT MLP (100, 50) Done...


In [ ]:
print("=== Mix Data BayesOPT with CV=5 ===")
for i in range(len(result_mix)):
  print(f"{i} -> {result_mix[i]}")

## Opt 3 Data

In [ ]:
## Bayesian OPT on 3 dataset ~250 for each
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from sklearn.model_selection import train_test_split
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="The objective has been evaluated at point")

# train and test split
output_col = df_main.loc[:, "DSS (MPa)":"STS (MPa)"].columns.to_list()
X_train, X_test, y_train, y_test = train_test_split(df_main.loc[:, :"Age (days)"], df_main.loc[:, "DSS (MPa)":"STS (MPa)"], test_size=0.3, random_state=42)

# normalize data
scaler = StandardScaler()
scaler.set_output(transform='pandas')
scaler.fit(X_train.loc[:, :"Age (days)"])
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
result_rf = []
for i in output_col:
  rf = RandomForestRegressor(random_state=42)
  rf_p = {'n_estimators': Integer(100, 500), 'criterion': Categorical(["squared_error", "absolute_error"])}
  opt = BayesSearchCV(rf, search_spaces = rf_p, cv = 5, n_iter = 10, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
  opt.fit(X_train, y_train[i])
  y_pred = opt.predict(X_test)
  result_rf.append({
      "Model": "Random Forest",
      "Output": i,
      "Best Params": opt.best_params_,
      "Test_MSE": mean_squared_error(y_test[i], y_pred)
  })
  print(f"RF {i} Done...")

mse_avg = 0
for i in result_rf:
  mse_avg = mse_avg + i['Test_MSE']

print(f"RF MSE {mse_avg / 3}")
for i in result_rf:
  print(i)

RF DSS (MPa) Done...
RF SSS (MPa) Done...
RF STS (MPa) Done...
RF MSE 1.1941150460482308
{'Model': 'Random Forest', 'Output': 'DSS (MPa)', 'Best Params': OrderedDict({'criterion': 'absolute_error', 'n_estimators': 468}), 'Test_MSE': 1.7884093701882156}
{'Model': 'Random Forest', 'Output': 'SSS (MPa)', 'Best Params': OrderedDict({'criterion': 'squared_error', 'n_estimators': 467}), 'Test_MSE': 1.6775828294334418}
{'Model': 'Random Forest', 'Output': 'STS (MPa)', 'Best Params': OrderedDict({'criterion': 'squared_error', 'n_estimators': 391}), 'Test_MSE': 0.11635293852303506}


In [ ]:
result_dt = []
dt = DecisionTreeRegressor(random_state=42)
dt_p = {'max_depth': Integer(1, 20)}
opt = BayesSearchCV(dt, search_spaces = dt_p, cv = 5, n_iter = 10, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
opt.fit(X_train, y_train)
y_pred = opt.predict(X_test)
result_dt.append({
    "Model": "Decision Tree",
    "Best Params": opt.best_params_,
    "Test_MSE": mean_squared_error(y_test, y_pred)
})
print(f"DT MSE {result_dt[0]['Test_MSE']}")
for i in result_dt:
  print(i)

DT MSE 2.3691036486435793
{'Model': 'Decision Tree', 'Best Params': OrderedDict({'max_depth': 9}), 'Test_MSE': 2.3691036486435793}


In [ ]:
# result_xgb = []
# xgb = xgboost.XGBRegressor()
# xgb_p = {'max_depth': Integer(3, 10)}
# opt = BayesSearchCV(xgb, search_spaces = xgb_p, cv = 5, n_iter = 10, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
# opt.fit(X_train, y_train)
# y_pred = opt.predict(X_test)
# result_xgb.append({
#     "Model": "Xgboost",
#     "Best Params": opt.best_params_,
#     "Test_MSE": mean_squared_error(y_test, y_pred)
# })
# print(f"Xgb MSE {result_xgb[0]['Test_MSE']}")
# for i in result_xgb:
#   print(i)

result_xgb = []
for i in output_col:
  xgb = xgboost.XGBRegressor(random_state=42)
  xgb_p = {'max_depth': Integer(3, 10)}
  opt = BayesSearchCV(xgb, search_spaces = xgb_p, cv = 5, n_iter = 10, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
  opt.fit(X_train, y_train[i])
  y_pred = opt.predict(X_test)
  result_xgb.append({
      "Model": "XGB",
      "Output": i,
      "Best Params": opt.best_params_,
      "Test_MSE": mean_squared_error(y_test[i], y_pred)
  })
  print(f"XGB {i} Done...")

mse_avg = 0
for i in result_xgb:
  mse_avg = mse_avg + i['Test_MSE']

print(f"XGB MSE {mse_avg / 3}")
for i in result_xgb:
  print(i)

XGB DSS (MPa) Done...
XGB SSS (MPa) Done...
XGB STS (MPa) Done...
XGB MSE 0.8882475493895896
{'Model': 'XGB', 'Output': 'DSS (MPa)', 'Best Params': OrderedDict({'max_depth': 3}), 'Test_MSE': 1.5213892004354947}
{'Model': 'XGB', 'Output': 'SSS (MPa)', 'Best Params': OrderedDict({'max_depth': 3}), 'Test_MSE': 1.1009964834461883}
{'Model': 'XGB', 'Output': 'STS (MPa)', 'Best Params': OrderedDict({'max_depth': 3}), 'Test_MSE': 0.042356964287085554}


In [ ]:
result_knn = []
knn = KNeighborsRegressor()
knn_p = {'n_neighbors': Integer(5, 11), 'weights': Categorical(['uniform', 'distance'])}
opt = BayesSearchCV(knn, search_spaces = knn_p, cv = 5, n_iter = 10, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
opt.fit(X_train, y_train)
y_pred = opt.predict(X_test)
result_knn.append({
    "Model": "KNN",
    "Best Params": opt.best_params_,
    "Test_MSE": mean_squared_error(y_test, y_pred)
})
print(f"KNN MSE {result_knn[0]['Test_MSE']}")
for i in result_knn:
  print(i)

KNN MSE 5.208887158996025
{'Model': 'KNN', 'Best Params': OrderedDict({'n_neighbors': 7, 'weights': 'distance'}), 'Test_MSE': 5.208887158996025}


In [ ]:
result_svr = []
for i in output_col:
  svr = SVR()
  svr_p = {'kernel': Categorical(['linear', 'poly', 'rbf']), 'C': Real(0.01, 1.0, prior='log-uniform')}
  opt = BayesSearchCV(svr, search_spaces = svr_p, cv = 5, n_iter = 10, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
  opt.fit(X_train, y_train[i])
  y_pred = opt.predict(X_test)
  result_svr.append({
      "Model": "SVR",
      "Output": i,
      "Best Params": opt.best_params_,
      "Test_MSE": mean_squared_error(y_test[i], y_pred)
  })
  print(f"SVR {i} Done...")

mse_avg = 0
for i in result_svr:
  mse_avg = mse_avg + i['Test_MSE']

print(f"SVR MSE {mse_avg / 3}")
for i in result_svr:
  print(i)

SVR DSS (MPa) Done...
SVR SSS (MPa) Done...
SVR STS (MPa) Done...
SVR MSE 4.1076599682717205
{'Model': 'SVR', 'Output': 'DSS (MPa)', 'Best Params': OrderedDict({'C': 0.42149456283334996, 'kernel': 'linear'}), 'Test_MSE': 3.9494239126998076}
{'Model': 'SVR', 'Output': 'SSS (MPa)', 'Best Params': OrderedDict({'C': 0.42149456283334996, 'kernel': 'linear'}), 'Test_MSE': 8.110671610838612}
{'Model': 'SVR', 'Output': 'STS (MPa)', 'Best Params': OrderedDict({'C': 0.8146216961026963, 'kernel': 'rbf'}), 'Test_MSE': 0.26288438127674346}


In [ ]:
result_el = []
for i in output_col:
  elasticnet = ElasticNet(random_state=42)
  elasticnet_p = {'l1_ratio': Real(0.0, 1.0), 'alpha': Real(0.01, 1.0, prior='log-uniform')}
  opt = BayesSearchCV(elasticnet, search_spaces = elasticnet_p, cv = 5, n_iter = 10, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
  opt.fit(X_train, y_train[i])
  y_pred = opt.predict(X_test)
  result_el.append({
      "Model": "Elasticnet",
      "Output": i,
      "Best Params": opt.best_params_,
      "Test_MSE": mean_squared_error(y_test[i], y_pred)
  })
  print(f"Elastic {i} Done...")

mse_avg = 0
for i in result_el:
  mse_avg = mse_avg + i['Test_MSE']

print(f"Elastic MSE {mse_avg / 3}")
for i in result_el:
  print(i)

Elastic DSS (MPa) Done...
Elastic SSS (MPa) Done...
Elastic STS (MPa) Done...
Elastic MSE 4.493264231603144
{'Model': 'Elasticnet', 'Output': 'DSS (MPa)', 'Best Params': OrderedDict({'alpha': 0.010168587136004645, 'l1_ratio': 0.8178300196637516}), 'Test_MSE': 4.015008012069639}
{'Model': 'Elasticnet', 'Output': 'SSS (MPa)', 'Best Params': OrderedDict({'alpha': 0.010168587136004645, 'l1_ratio': 0.8178300196637516}), 'Test_MSE': 8.712405085398814}
{'Model': 'Elasticnet', 'Output': 'STS (MPa)', 'Best Params': OrderedDict({'alpha': 0.010168587136004645, 'l1_ratio': 0.8178300196637516}), 'Test_MSE': 0.7523795973409785}


In [ ]:
result_gbr = []
for i in output_col:
  gbr = GradientBoostingRegressor(random_state=42)
  if i == "SSS (MPa)":
    gbr_p = {'learning_rate': Categorical([0.1, 0.001])}
  else:
    gbr_p = {'n_estimators': Integer(100, 500), 'learning_rate': Real(0.01, 0.3, prior='log-uniform')}
  opt = BayesSearchCV(gbr, search_spaces = gbr_p, cv = 5, n_iter = 10, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
  opt.fit(X_train, y_train[i])
  y_pred = opt.predict(X_test)
  result_gbr.append({
      "Model": "Gradient Boosting Reg",
      "Output": i,
      "Best Params": opt.best_params_,
      "Test_MSE": mean_squared_error(y_test[i], y_pred)
  })
  print(f"GBR {i} Done...")

mse_avg = 0
for i in result_gbr:
  mse_avg = mse_avg + i['Test_MSE']

print(f"Gradient Boosting Reg MSE {mse_avg / 3}")
for i in result_gbr:
  print(i)

GBR DSS (MPa) Done...
GBR SSS (MPa) Done...
GBR STS (MPa) Done...
Gradient Boosting Reg MSE 0.773698886098498
{'Model': 'Gradient Boosting Reg', 'Output': 'DSS (MPa)', 'Best Params': OrderedDict({'learning_rate': 0.08156447333825878, 'n_estimators': 410}), 'Test_MSE': 1.3403667557349574}
{'Model': 'Gradient Boosting Reg', 'Output': 'SSS (MPa)', 'Best Params': OrderedDict({'learning_rate': 0.1}), 'Test_MSE': 0.9359943376228934}
{'Model': 'Gradient Boosting Reg', 'Output': 'STS (MPa)', 'Best Params': OrderedDict({'learning_rate': 0.2578438884524839, 'n_estimators': 380}), 'Test_MSE': 0.044735564937643295}


In [ ]:
result_mlp = []
for i in output_col:
  r = []
  for j in [(100,), (128,64),(128,64,32), (100, 50)]:
    mlp = MLPRegressor(hidden_layer_sizes=j, random_state=4)
    mlp_p = {
        'activation': Categorical(['tanh', 'relu']),
        'learning_rate_init': Real(0.0001, 0.1, prior='log-uniform')
        }
    opt = BayesSearchCV(mlp, search_spaces = mlp_p, cv = 5, n_iter = 10, random_state = 42, scoring = 'neg_mean_squared_error', n_jobs = -1)
    opt.fit(X_train, y_train[i])
    y_pred = opt.predict(X_test)
    r.append({
        "Model": f"MLP {j}",
        "Output": i,
        "Best Params": opt.best_params_,
        "Test_MSE": mean_squared_error(y_test[i], y_pred)
    })
    print(f"MLP {j} - {i} Done...")
  result_mlp.append(r)

best_models = [min(group, key=lambda x: x['Test_MSE']) for group in result_mlp]

mse_avg = 0
for i in best_models:
  mse_avg = mse_avg + i['Test_MSE']

print(f"MLP MSE {mse_avg / 3}")
for i in best_models:
  print(i)

MLP (100,) - DSS (MPa) Done...
MLP (128, 64) - DSS (MPa) Done...
MLP (128, 64, 32) - DSS (MPa) Done...
MLP (100, 50) - DSS (MPa) Done...
MLP (100,) - SSS (MPa) Done...
MLP (128, 64) - SSS (MPa) Done...
MLP (128, 64, 32) - SSS (MPa) Done...
MLP (100, 50) - SSS (MPa) Done...
MLP (100,) - STS (MPa) Done...
MLP (128, 64) - STS (MPa) Done...
MLP (128, 64, 32) - STS (MPa) Done...
MLP (100, 50) - STS (MPa) Done...
MLP MSE 0.8121419893588729
{'Model': 'MLP (128, 64)', 'Output': 'DSS (MPa)', 'Best Params': OrderedDict({'activation': 'relu', 'learning_rate_init': 0.044662788252242826}), 'Test_MSE': 1.2276589203843082}
{'Model': 'MLP (128, 64, 32)', 'Output': 'SSS (MPa)', 'Best Params': OrderedDict({'activation': 'relu', 'learning_rate_init': 0.044662788252242826}), 'Test_MSE': 1.1592658801567164}
{'Model': 'MLP (100,)', 'Output': 'STS (MPa)', 'Best Params': OrderedDict({'activation': 'relu', 'learning_rate_init': 0.044662788252242826}), 'Test_MSE': 0.04950116753559406}


## Final Results

In [ ]:
### Final

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
import xgboost
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

# train and test split
output_col = df_main.loc[:, "DSS (MPa)":"STS (MPa)"].columns.to_list()
X_train, X_test, y_train, y_test = train_test_split(df_main.loc[:, :"Age (days)"], df_main.loc[:, "DSS (MPa)":"STS (MPa)"], test_size=0.3, random_state=42)

# normalize data
scaler = StandardScaler()
scaler.set_output(transform='pandas')
scaler.fit(X_train.loc[:, :"Age (days)"])
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model_dss = GradientBoostingRegressor(learning_rate= 0.08156447333825878, n_estimators= 410, random_state=42)
model_sss = GradientBoostingRegressor(learning_rate= 0.1, random_state=42)
model_sts = GradientBoostingRegressor(learning_rate= 0.2578438884524839, n_estimators= 380, random_state=42)

model_dss.fit(X_train, y_train['DSS (MPa)'])
model_sss.fit(X_train, y_train['SSS (MPa)'])
model_sts.fit(X_train, y_train['STS (MPa)'])

overall_r2 = 0
overall_mae = 0
overall_mse = 0
overall_rmse = 0

train_r2 = 0
train_mae = 0
train_mse = 0
train_rmse = 0

for i in [('DSS (MPa)', model_dss), ('SSS (MPa)', model_sss), ('STS (MPa)', model_sts)]:
  res = i[1].predict(X_test)
  r2 = r2_score(y_test[i[0]], res)
  mae = mean_absolute_error(y_test[i[0]], res)
  mse = mean_squared_error(y_test[i[0]], res)
  rmse = np.sqrt(mse)
  print(f"GBR {i[0]} (Test)  R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")
  overall_r2 = overall_r2 + r2
  overall_mae = overall_mae + mae
  overall_mse = overall_mse + mse
  overall_rmse = overall_rmse + rmse

  res = i[1].predict(X_train)
  r2 = r2_score(y_train[i[0]], res)
  mae = mean_absolute_error(y_train[i[0]], res)
  mse = mean_squared_error(y_train[i[0]], res)
  rmse = np.sqrt(mse)
  train_r2 = train_r2 + r2
  train_mae = train_mae + mae
  train_mse = train_mse + mse
  train_rmse = train_rmse + rmse

  print(f"GBR {i[0]} (Train) R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")

print(f"Overall Test R2:{overall_r2/3:.4f} MAE:{overall_mae/3:.4f} MSE:{overall_mse/3:.4f} RMSE:{overall_rmse/3:.4f}")
print(f"Overall Train R2:{train_r2/3:.4f} MAE:{train_mae/3:.4f} MSE:{train_mse/3:.4f} RMSE:{train_rmse/3:.4f}")

GBR DSS (MPa) (Test)  R2:0.9322 MAE:0.8371 MSE:1.3404 RMSE:1.1577
GBR DSS (MPa) (Train) R2:0.9927 MAE:0.3174 MSE:0.1628 RMSE:0.4035
GBR SSS (MPa) (Test)  R2:0.9770 MAE:0.7991 MSE:0.9360 RMSE:0.9675
GBR SSS (MPa) (Train) R2:0.9919 MAE:0.4805 MSE:0.3645 RMSE:0.6037
GBR STS (MPa) (Test)  R2:0.9809 MAE:0.1567 MSE:0.0447 RMSE:0.2115
GBR STS (MPa) (Train) R2:0.9994 MAE:0.0276 MSE:0.0014 RMSE:0.0379
Overall Test R2:0.9634 MAE:0.5976 MSE:0.7737 RMSE:0.7789
Overall Train R2:0.9947 MAE:0.2752 MSE:0.1762 RMSE:0.3484


In [ ]:
model_dss = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', learning_rate_init=0.044662788252242826, random_state=4)
model_sss = MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', learning_rate_init=0.044662788252242826, random_state=4)
model_sts = MLPRegressor(hidden_layer_sizes=(100,), activation='relu', learning_rate_init=0.044662788252242826, random_state=4)

model_dss.fit(X_train, y_train['DSS (MPa)'])
model_sss.fit(X_train, y_train['SSS (MPa)'])
model_sts.fit(X_train, y_train['STS (MPa)'])

overall_r2 = 0
overall_mae = 0
overall_mse = 0
overall_rmse = 0

train_r2 = 0
train_mae = 0
train_mse = 0
train_rmse = 0

for i in [('DSS (MPa)', model_dss), ('SSS (MPa)', model_sss), ('STS (MPa)', model_sts)]:
  res = i[1].predict(X_test)
  r2 = r2_score(y_test[i[0]], res)
  mae = mean_absolute_error(y_test[i[0]], res)
  mse = mean_squared_error(y_test[i[0]], res)
  rmse = np.sqrt(mse)
  print(f"MLP {i[0]} (Test)  R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")
  overall_r2 = overall_r2 + r2
  overall_mae = overall_mae + mae
  overall_mse = overall_mse + mse
  overall_rmse = overall_rmse + rmse

  res = i[1].predict(X_train)
  r2 = r2_score(y_train[i[0]], res)
  mae = mean_absolute_error(y_train[i[0]], res)
  mse = mean_squared_error(y_train[i[0]], res)
  rmse = np.sqrt(mse)
  train_r2 = train_r2 + r2
  train_mae = train_mae + mae
  train_mse = train_mse + mse
  train_rmse = train_rmse + rmse
  print(f"MLP {i[0]} (Train) R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")

print(f"Overall Test R2:{overall_r2/3:.4f} MAE:{overall_mae/3:.4f} MSE:{overall_mse/3:.4f} RMSE:{overall_rmse/3:.4f}")
print(f"Overall Train R2:{train_r2/3:.4f} MAE:{train_mae/3:.4f} MSE:{train_mse/3:.4f} RMSE:{train_rmse/3:.4f}")

MLP DSS (MPa) (Test)  R2:0.9379 MAE:0.8680 MSE:1.2277 RMSE:1.1080
MLP DSS (MPa) (Train) R2:0.9885 MAE:0.3829 MSE:0.2562 RMSE:0.5061
MLP SSS (MPa) (Test)  R2:0.9716 MAE:0.8256 MSE:1.1593 RMSE:1.0767
MLP SSS (MPa) (Train) R2:0.9975 MAE:0.2415 MSE:0.1109 RMSE:0.3331
MLP STS (MPa) (Test)  R2:0.9789 MAE:0.1634 MSE:0.0495 RMSE:0.2225
MLP STS (MPa) (Train) R2:0.9962 MAE:0.0682 MSE:0.0089 RMSE:0.0943
Overall Test R2:0.9628 MAE:0.6190 MSE:0.8121 RMSE:0.8024
Overall Train R2:0.9941 MAE:0.2309 MSE:0.1253 RMSE:0.3112


In [ ]:
model_dss = RandomForestRegressor(criterion='absolute_error', n_estimators=468, random_state=42)
model_sss = RandomForestRegressor(criterion='squared_error', n_estimators=467, random_state=42)
model_sts = RandomForestRegressor(criterion='squared_error', n_estimators=391, random_state=42)

model_dss.fit(X_train, y_train['DSS (MPa)'])
model_sss.fit(X_train, y_train['SSS (MPa)'])
model_sts.fit(X_train, y_train['STS (MPa)'])

overall_r2 = 0
overall_mae = 0
overall_mse = 0
overall_rmse = 0

train_r2 = 0
train_mae = 0
train_mse = 0
train_rmse = 0

for i in [('DSS (MPa)', model_dss), ('SSS (MPa)', model_sss), ('STS (MPa)', model_sts)]:
  res = i[1].predict(X_test)
  r2 = r2_score(y_test[i[0]], res)
  mae = mean_absolute_error(y_test[i[0]], res)
  mse = mean_squared_error(y_test[i[0]], res)
  rmse = np.sqrt(mse)
  print(f"RF {i[0]} (Test)  R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")
  overall_r2 = overall_r2 + r2
  overall_mae = overall_mae + mae
  overall_mse = overall_mse + mse
  overall_rmse = overall_rmse + rmse

  res = i[1].predict(X_train)
  r2 = r2_score(y_train[i[0]], res)
  mae = mean_absolute_error(y_train[i[0]], res)
  mse = mean_squared_error(y_train[i[0]], res)
  rmse = np.sqrt(mse)
  train_r2 = train_r2 + r2
  train_mae = train_mae + mae
  train_mse = train_mse + mse
  train_rmse = train_rmse + rmse
  print(f"RF {i[0]} (Train) R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")

print(f"Overall Test R2:{overall_r2/3:.4f} MAE:{overall_mae/3:.4f} MSE:{overall_mse/3:.4f} RMSE:{overall_rmse/3:.4f}")
print(f"Overall Train R2:{train_r2/3:.4f} MAE:{train_mae/3:.4f} MSE:{train_mse/3:.4f} RMSE:{train_rmse/3:.4f}")

RF DSS (MPa) (Test)  R2:0.9095 MAE:1.0518 MSE:1.7884 RMSE:1.3373
RF DSS (MPa) (Train) R2:0.9862 MAE:0.4292 MSE:0.3078 RMSE:0.5548
RF SSS (MPa) (Test)  R2:0.9588 MAE:0.9852 MSE:1.6776 RMSE:1.2952
RF SSS (MPa) (Train) R2:0.9921 MAE:0.4461 MSE:0.3545 RMSE:0.5954
RF STS (MPa) (Test)  R2:0.9503 MAE:0.2694 MSE:0.1164 RMSE:0.3411
RF STS (MPa) (Train) R2:0.9932 MAE:0.0966 MSE:0.0160 RMSE:0.1265
Overall Test R2:0.9396 MAE:0.7688 MSE:1.1941 RMSE:0.9912
Overall Train R2:0.9905 MAE:0.3240 MSE:0.2261 RMSE:0.4256


In [ ]:
model = DecisionTreeRegressor(max_depth=9, random_state=42)
model.fit(X_train, y_train)

overall_r2 = 0
overall_mae = 0
overall_mse = 0
overall_rmse = 0

train_r2 = 0
train_mae = 0
train_mse = 0
train_rmse = 0

y_1 = model.predict(X_test)
for i, output in enumerate(['DSS (MPa)', 'SSS (MPa)', 'STS (MPa)']):
  r2 = r2_score(y_test[output], y_1[:, i])
  mae = mean_absolute_error(y_test[output], y_1[:, i])
  mse = mean_squared_error(y_test[output], y_1[:, i])
  rmse = np.sqrt(mse)
  print(f"DT {output} (Test)  R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")
  overall_r2 = overall_r2 + r2
  overall_mae = overall_mae + mae
  overall_mse = overall_mse + mse
  overall_rmse = overall_rmse + rmse

y_2 = model.predict(X_train)
for i, output in enumerate(['DSS (MPa)', 'SSS (MPa)', 'STS (MPa)']):
  r2 = r2_score(y_train[output], y_2[:, i])
  mae = mean_absolute_error(y_train[output], y_2[:, i])
  mse = mean_squared_error(y_train[output], y_2[:, i])
  rmse = np.sqrt(mse)
  train_r2 = train_r2 + r2
  train_mae = train_mae + mae
  train_mse = train_mse + mse
  train_rmse = train_rmse + rmse
  print(f"DT {output} (Train) R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")

print(f"Overall Test R2:{overall_r2/3:.4f} MAE:{overall_mae/3:.4f} MSE:{overall_mse/3:.4f} RMSE:{overall_rmse/3:.4f}")
print(f"Overall Train R2:{train_r2/3:.4f} MAE:{train_mae/3:.4f} MSE:{train_mse/3:.4f} RMSE:{train_rmse/3:.4f}")

DT DSS (MPa) (Test)  R2:0.8229 MAE:1.3905 MSE:3.5005 RMSE:1.8710
DT SSS (MPa) (Test)  R2:0.9180 MAE:1.3364 MSE:3.3417 RMSE:1.8280
DT STS (MPa) (Test)  R2:0.8868 MAE:0.4325 MSE:0.2651 RMSE:0.5149
DT DSS (MPa) (Train) R2:0.9976 MAE:0.0803 MSE:0.0539 RMSE:0.2322
DT SSS (MPa) (Train) R2:0.9993 MAE:0.0581 MSE:0.0295 RMSE:0.1718
DT STS (MPa) (Train) R2:0.9929 MAE:0.0423 MSE:0.0167 RMSE:0.1293
Overall Test R2:0.8759 MAE:1.0531 MSE:2.3691 RMSE:1.4046
Overall Train R2:0.9966 MAE:0.0602 MSE:0.0334 RMSE:0.1777


In [ ]:
# model = xgboost.XGBRegressor(max_depth=3, random_state=42)
# model.fit(X_train, y_train)

# overall_r2 = 0
# overall_mae = 0
# overall_mse = 0
# overall_rmse = 0

# train_r2 = 0
# train_mae = 0
# train_mse = 0
# train_rmse = 0

# y_1 = model.predict(X_test)
# for i, output in enumerate(['DSS (MPa)', 'SSS (MPa)', 'STS (MPa)']):
#   r2 = r2_score(y_test[output], y_1[:, i])
#   mae = mean_absolute_error(y_test[output], y_1[:, i])
#   mse = mean_squared_error(y_test[output], y_1[:, i])
#   rmse = np.sqrt(mse)
#   print(f"XGB {output} (Test)  R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")
#   overall_r2 = overall_r2 + r2
#   overall_mae = overall_mae + mae
#   overall_mse = overall_mse + mse
#   overall_rmse = overall_rmse + rmse

# y_2 = model.predict(X_train)
# for i, output in enumerate(['DSS (MPa)', 'SSS (MPa)', 'STS (MPa)']):
#   r2 = r2_score(y_train[output], y_2[:, i])
#   mae = mean_absolute_error(y_train[output], y_2[:, i])
#   mse = mean_squared_error(y_train[output], y_2[:, i])
#   rmse = np.sqrt(mse)
#   train_r2 = train_r2 + r2
#   train_mae = train_mae + mae
#   train_mse = train_mse + mse
#   train_rmse = train_rmse + rmse
#   print(f"XGB {output} (Train) R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")

# print(f"Overall Test R2:{overall_r2/3:.4f} MAE:{overall_mae/3:.4f} MSE:{overall_mse/3:.4f} RMSE:{overall_rmse/3:.4f}")
# print(f"Overall Train R2:{train_r2/3:.4f} MAE:{train_mae/3:.4f} MSE:{train_mse/3:.4f} RMSE:{train_rmse/3:.4f}")

XGB DSS (MPa) (Test)  R2:0.9230 MAE:0.9112 MSE:1.5214 RMSE:1.2334
XGB SSS (MPa) (Test)  R2:0.9730 MAE:0.8581 MSE:1.1010 RMSE:1.0493
XGB STS (MPa) (Test)  R2:0.9819 MAE:0.1566 MSE:0.0424 RMSE:0.2058
XGB DSS (MPa) (Train) R2:0.9906 MAE:0.3588 MSE:0.2103 RMSE:0.4586
XGB SSS (MPa) (Train) R2:0.9956 MAE:0.3468 MSE:0.1993 RMSE:0.4464
XGB STS (MPa) (Train) R2:0.9971 MAE:0.0595 MSE:0.0069 RMSE:0.0832
Overall Test R2:0.9593 MAE:0.6420 MSE:0.8882 RMSE:0.8295
Overall Train R2:0.9944 MAE:0.2550 MSE:0.1388 RMSE:0.3294


In [ ]:
model_dss = xgboost.XGBRegressor(max_depth=3, random_state=42)
model_sss = xgboost.XGBRegressor(max_depth=3, random_state=42)
model_sts = xgboost.XGBRegressor(max_depth=3, random_state=42)

model_dss.fit(X_train, y_train['DSS (MPa)'])
model_sss.fit(X_train, y_train['SSS (MPa)'])
model_sts.fit(X_train, y_train['STS (MPa)'])

overall_r2 = 0
overall_mae = 0
overall_mse = 0
overall_rmse = 0

train_r2 = 0
train_mae = 0
train_mse = 0
train_rmse = 0

for i in [('DSS (MPa)', model_dss), ('SSS (MPa)', model_sss), ('STS (MPa)', model_sts)]:
  res = i[1].predict(X_test)
  r2 = r2_score(y_test[i[0]], res)
  mae = mean_absolute_error(y_test[i[0]], res)
  mse = mean_squared_error(y_test[i[0]], res)
  rmse = np.sqrt(mse)
  print(f"XGB {i[0]} (Test)  R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")
  overall_r2 = overall_r2 + r2
  overall_mae = overall_mae + mae
  overall_mse = overall_mse + mse
  overall_rmse = overall_rmse + rmse

  res = i[1].predict(X_train)
  r2 = r2_score(y_train[i[0]], res)
  mae = mean_absolute_error(y_train[i[0]], res)
  mse = mean_squared_error(y_train[i[0]], res)
  rmse = np.sqrt(mse)
  train_r2 = train_r2 + r2
  train_mae = train_mae + mae
  train_mse = train_mse + mse
  train_rmse = train_rmse + rmse
  print(f"XGB {i[0]} (Train) R2:{r2:.4f} MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f}")

print(f"Overall Test R2:{overall_r2/3:.4f} MAE:{overall_mae/3:.4f} MSE:{overall_mse/3:.4f} RMSE:{overall_rmse/3:.4f}")
print(f"Overall Train R2:{train_r2/3:.4f} MAE:{train_mae/3:.4f} MSE:{train_mse/3:.4f} RMSE:{train_rmse/3:.4f}")

XGB DSS (MPa) (Test)  R2:0.9230 MAE:0.9112 MSE:1.5214 RMSE:1.2334
XGB DSS (MPa) (Train) R2:0.9906 MAE:0.3588 MSE:0.2103 RMSE:0.4586
XGB SSS (MPa) (Test)  R2:0.9730 MAE:0.8581 MSE:1.1010 RMSE:1.0493
XGB SSS (MPa) (Train) R2:0.9956 MAE:0.3468 MSE:0.1993 RMSE:0.4464
XGB STS (MPa) (Test)  R2:0.9819 MAE:0.1566 MSE:0.0424 RMSE:0.2058
XGB STS (MPa) (Train) R2:0.9971 MAE:0.0595 MSE:0.0069 RMSE:0.0832
Overall Test R2:0.9593 MAE:0.6420 MSE:0.8882 RMSE:0.8295
Overall Train R2:0.9944 MAE:0.2550 MSE:0.1388 RMSE:0.3294


In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Example data (replace with your actual metrics)
# models = ['GBR', 'MLP', 'RF', 'DT', 'XGB']
# metrics = ['R2', 'MAE', 'MSE', 'RMSE']

# # Each entry: [R2, MAE, MSE, RMSE] for train and test
# train_scores = [
#     [0.9947, 0.2752, 0.1762, 0.3484],
#     [0.9941, 0.2309, 0.1253, 0.3112],
#     [0.9905, 0.3240, 0.2261, 0.4256],
#     [0.9966, 0.0602, 0.0334, 0.1777],
#     [0.9944, 0.2550, 0.1388, 0.3294]
# ]

# test_scores = [
#     [0.9634, 0.5976, 0.7737, 0.7789],
#     [0.9628, 0.6190, 0.8121, 0.8024],
#     [0.9396, 0.7688, 1.1941, 0.9912],
#     [0.8759, 1.0531, 2.3691, 1.4046],
#     [0.9593, 0.6420, 0.8882, 0.8295]
# ]

# # Set up radar chart
# num_vars = len(metrics)
# angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
# angles += angles[:1]  # close the loop

# fig, axes = plt.subplots(1, 5, subplot_kw=dict(polar=True), figsize=(20, 4), dpi=200)

# for i, ax in enumerate(axes):
#     train_values = train_scores[i] + train_scores[i][:1]
#     test_values = test_scores[i] + test_scores[i][:1]

#     ax.plot(angles, train_values, color='orange', linewidth=2, label='Train')
#     ax.fill(angles, train_values, color='orange', alpha=0.25)

#     ax.plot(angles, test_values, color='royalblue', linewidth=2, label='Test')
#     ax.fill(angles, test_values, color='royalblue', alpha=0.1)

#     ax.set_xticks(angles[:-1])
#     ax.set_xticklabels(metrics)
#     ax.set_title(models[i], size=13, weight='bold', pad=25)
#     # ax.set_title(models[i], size=13, weight='bold')
#     ax.grid(True)
#     ax.patch.set_facecolor('gainsboro')

# # Add legend once
# axes[0].legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
# plt.tight_layout()
# plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example data
models = ['GBR', 'MLPNN', 'RF', 'DT', 'XGB']
metrics = ['R2', 'MAE', 'MSE', 'RMSE']

train_scores = [
    [0.9947, 0.2752, 0.1762, 0.3484],
    [0.9941, 0.2309, 0.1253, 0.3112],
    [0.9905, 0.3240, 0.2261, 0.4256],
    [0.9966, 0.0602, 0.0334, 0.1777],
    [0.9944, 0.2550, 0.1388, 0.3294]
]

test_scores = [
    [0.9634, 0.5976, 0.7737, 0.7789],
    [0.9628, 0.6190, 0.8121, 0.8024],
    [0.9396, 0.7688, 1.1941, 0.9912],
    [0.8759, 1.0531, 2.3691, 1.4046],
    [0.9593, 0.6420, 0.8882, 0.8295]
]

# Radar chart setup
num_vars = len(metrics)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # close the loop

# Create figure
fig = plt.figure(figsize=(18, 10), dpi=1200)

# Positions for first row (3 charts)
first_row_pos = [0.05, 0.55, 0.28, 0.4], [0.36, 0.55, 0.28, 0.4], [0.67, 0.55, 0.28, 0.4]

# Positions for second row (2 charts centered)
second_row_pos = [0.18, 0.05, 0.28, 0.4], [0.53, 0.05, 0.28, 0.4]

# Plot first row
for i in range(3):
    ax = fig.add_axes(first_row_pos[i], polar=True)
    train_values = train_scores[i] + train_scores[i][:1]
    test_values = test_scores[i] + test_scores[i][:1]

    ax.plot(angles, train_values, color='orange', linewidth=2, label='Train')
    ax.fill(angles, train_values, color='orange', alpha=0.25)
    ax.plot(angles, test_values, color='royalblue', linewidth=2, label='Test')
    ax.fill(angles, test_values, color='royalblue', alpha=0.1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, fontsize=12, fontweight='bold')
    ax.tick_params(labelsize=14)
    ax.set_title(models[i], size=20, weight='bold', pad=30)
    ax.grid(True)
    ax.patch.set_facecolor('whitesmoke')

# Plot second row
for i in range(2):
    ax = fig.add_axes(second_row_pos[i], polar=True)
    j = i + 3  # index in models
    train_values = train_scores[j] + train_scores[j][:1]
    test_values = test_scores[j] + test_scores[j][:1]

    ax.plot(angles, train_values, color='orange', linewidth=2, label='Train')
    ax.fill(angles, train_values, color='orange', alpha=0.25)
    ax.plot(angles, test_values, color='royalblue', linewidth=2, label='Test')
    ax.fill(angles, test_values, color='royalblue', alpha=0.1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, fontsize=12, fontweight='bold')
    ax.tick_params(labelsize=14)
    ax.set_title(models[j], size=20, weight='bold', pad=30)
    ax.grid(True)
    ax.patch.set_facecolor('whitesmoke')

# Add legend once
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=18)

plt.show()


In [ ]:
import shap

model_dss = GradientBoostingRegressor(learning_rate= 0.08156447333825878, n_estimators= 410, random_state=42)
model_sss = GradientBoostingRegressor(learning_rate= 0.1, random_state=42)
model_sts = GradientBoostingRegressor(learning_rate= 0.2578438884524839, n_estimators= 380, random_state=42)

model_dss.fit(X_train, y_train['DSS (MPa)'])
model_sss.fit(X_train, y_train['SSS (MPa)'])
model_sts.fit(X_train, y_train['STS (MPa)'])

explainer_dss = shap.Explainer(model_dss)
shap_values_dss = explainer_dss(X_test)

explainer_sss = shap.Explainer(model_sss)
shap_values_sss = explainer_sss(X_test)

explainer_sts = shap.Explainer(model_sts)
shap_values_sts = explainer_sts(X_test)

In [ ]:
# shap.plots.beeswarm(shap_values_dss, show=False)
# plt.gcf().set_dpi(1200)
# plt.show()

shap.plots.bar(shap_values_dss, show=False)
plt.gcf().set_dpi(800)
plt.show()

In [ ]:
# shap.plots.beeswarm(shap_values_sss, show=False)
# plt.gcf().set_dpi(1200)
# plt.show()

shap.plots.bar(shap_values_sss, show=False)
plt.gcf().set_dpi(800)
plt.show()

In [ ]:
# shap.plots.beeswarm(shap_values_sts, show=False)
# plt.gcf().set_dpi(1200)
# plt.show()

shap.plots.bar(shap_values_sts, show=False)
plt.gcf().set_dpi(800)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(df_main.loc[:, :"Age (days)"], df_main.loc[:, "DSS (MPa)":"STS (MPa)"], test_size=0.3, random_state=42)

model_dss = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(learning_rate= 0.08156447333825878, n_estimators= 410, random_state=42))
])
model_sss = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(learning_rate= 0.1, random_state=42))
])
model_sts = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(learning_rate= 0.2578438884524839, n_estimators= 380, random_state=42))
])

# model_dss = GradientBoostingRegressor(learning_rate= 0.08156447333825878, n_estimators= 410, random_state=42)
# model_sss = GradientBoostingRegressor(learning_rate= 0.1, random_state=42)
# model_sts = GradientBoostingRegressor(learning_rate= 0.2578438884524839, n_estimators= 380, random_state=42)

model_dss.fit(X_train, y_train['DSS (MPa)'])
model_sss.fit(X_train, y_train['SSS (MPa)'])
model_sts.fit(X_train, y_train['STS (MPa)'])


models = [model_dss, model_sss, model_sts]
feature_names = X_train.columns.to_list()
output_names = ["DSS (MPa)", "SSS (MPa)", "STS (MPa)"]

fig, axes = plt.subplots(3, 7, figsize=(28, 12), dpi=600)  # 3 rows × 7 columns

for i, model in enumerate(models):
    for j, feature in enumerate(range(7)):
        PartialDependenceDisplay.from_estimator(
            model, X_train, [feature], ax=axes[i, j], feature_names=feature_names, line_kw={"color": "red", "linewidth": 2, "linestyle": "--"}
        )
        axes[i, j].set_title(f"{output_names[i]}")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model_dss = GradientBoostingRegressor(learning_rate= 0.08156447333825878, n_estimators= 410, random_state=42)
model_sss = GradientBoostingRegressor(learning_rate= 0.1, random_state=42)
model_sts = GradientBoostingRegressor(learning_rate= 0.2578438884524839, n_estimators= 380, random_state=42)

# model_dss = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', learning_rate_init=0.044662788252242826, random_state=4)
# model_sss = MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', learning_rate_init=0.044662788252242826, random_state=4)
# model_sts = MLPRegressor(hidden_layer_sizes=(100,), activation='relu', learning_rate_init=0.044662788252242826, random_state=4)

# model_dss = RandomForestRegressor(criterion='absolute_error', n_estimators=468, random_state=42)
# model_sss = RandomForestRegressor(criterion='squared_error', n_estimators=467, random_state=42)
# model_sts = RandomForestRegressor(criterion='squared_error', n_estimators=391, random_state=42)

# model_dss = xgboost.XGBRegressor(max_depth=3, random_state=42)
# model_sss = xgboost.XGBRegressor(max_depth=3, random_state=42)
# model_sts = xgboost.XGBRegressor(max_depth=3, random_state=42)

model_dss.fit(X_train, y_train['DSS (MPa)'])
model_sss.fit(X_train, y_train['SSS (MPa)'])
model_sts.fit(X_train, y_train['STS (MPa)'])

pred1 = model_dss.predict(X_test)
pred2 = model_sss.predict(X_test)
pred3 = model_sts.predict(X_test)

actuals = [y_test['DSS (MPa)'].values, y_test['SSS (MPa)'].values, y_test['STS (MPa)'].values]
preds = [pred1, pred2, pred3]
titles = ['DSS (MPa)','SSS (MPa)','STS (MPa)']

plt.figure(figsize=(18,5), dpi=1200)

for i in range(3):
    actual = actuals[i]
    pred = preds[i]

    plt.subplot(1, 3, i+1)

    # Plot actual vs predicted
    plt.plot(actual, label='Actual', linewidth=2, color='royalblue')
    plt.plot(pred, label='Predicted', linewidth=2, color='orange')

    # Shaded error region
    errors = np.abs(actual - pred)
    plt.fill_between(range(len(actual)),
                     actual - errors,
                     actual + errors,
                     color='gray', alpha=0.2)

    # Mean values in title
    mean_actual = np.mean(actual)
    mean_pred = np.mean(pred)
    plt.title(f"Mean Actual = {mean_actual:.2f}, Mean Predicted = {mean_pred:.2f}",
              fontsize=16, fontweight='bold')

    # Add axis labels
    plt.xlabel('Test set samples', fontsize=15)
    plt.ylabel(titles[i], fontsize=15)

    # Increase font size of numbers on axes
    plt.tick_params(axis='both', labelsize=14)

    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()

plt.tight_layout()
plt.show()



In [ ]:
model = DecisionTreeRegressor(max_depth=9, random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

actuals = [y_test['DSS (MPa)'].values, y_test['SSS (MPa)'].values, y_test['STS (MPa)'].values]
preds = pred
titles = ['DSS (MPa)','SSS (MPa)','STS (MPa)']

plt.figure(figsize=(18,5), dpi=1200)

for i in range(3):
    actual = actuals[i]
    pred = preds[:, i]

    plt.subplot(1, 3, i+1)

    # Plot actual vs predicted
    plt.plot(actual, label='Actual', linewidth=2, color='royalblue')
    plt.plot(pred, label='Predicted', linewidth=2, color='orange')

    # Shaded error region
    errors = np.abs(actual - pred)
    plt.fill_between(range(len(actual)),
                     actual - errors,
                     actual + errors,
                     color='gray', alpha=0.2)

    # Mean values in title
    mean_actual = np.mean(actual)
    mean_pred = np.mean(pred)
    plt.title(f"Mean Actual = {mean_actual:.2f}, Mean Predicted = {mean_pred:.2f}",
              fontsize=16, fontweight='bold')

    # Add axis labels
    plt.xlabel('Test set samples', fontsize=15)
    plt.ylabel(titles[i], fontsize=15)

    # Increase font size of numbers on axes
    plt.tick_params(axis='both', labelsize=14)

    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()

plt.tight_layout()
plt.show()